# Run the pretrained small model on random data:


# 

In [20]:
import logging
from datetime import datetime

import matplotlib.pyplot as plt
import torch

from aurora import AuroraSmall, Batch, Metadata, rollout

In [21]:
# Allow custom grid dimensions
def create_aurora_batch(grid_height=17, grid_width=32, num_levels=13):
    """
    Create a sample batch for the Aurora model with customizable grid dimensions.

    Args:
        grid_height (int): Height of the grid
        grid_width (int): Width of the grid
        num_levels (int): Number of atmospheric levels

    Returns:
        Batch: An Aurora batch with random data
    """

    # For custom number of levels, generate reasonable pressure levels
    # between 1000 hPa (surface) and 100 hPa (upper atmosphere)
    # pressure_levels = tuple(np.linspace(100, 1000, num_levels, dtype=int))
    pressure_levels = tuple(range(100, 1001, 50))
    logging.info(f"Pressure levels: {pressure_levels}")

    # Create a batch with the specified dimensions
    batch = Batch(
        # surf_vars={k: torch.randn(1, 2, grid_height, grid_width) for k in ("2t", "10u", "10v", "msl")},
        surf_vars={k: torch.randn(1, 2, grid_height, grid_width) for k in ("2t", "10u", "10v")},
        # static_vars={k: torch.randn(grid_height, grid_width) for k in ("lsm", "z", "slt")},
        static_vars={k: torch.randn(grid_height, grid_width) for k in ("z")},
        atmos_vars={
            k: torch.randn(1, 2, num_levels, grid_height, grid_width)
            for k in ("z", "u", "v", "t", "q")
        },
        metadata=Metadata(
            lat=torch.linspace(40, -90, grid_height),
            lon=torch.linspace(0, 260, grid_width + 1)[:-1],
            time=(datetime(2020, 6, 1, 12, 0),),
            atmos_levels=pressure_levels,
        ),
    )

    return batch

In [22]:
def run_aurora_model(batch, steps=3):
    """
    Run Aurora model with the provided batch and optionally perform multi-step rollout.

    Args:
        batch (Batch): The input batch for Aurora model
        steps (int): Number of rollout steps to predict

    Returns:
        tuple: (prediction, rollout_predictions) where prediction is the immediate forecast
               and rollout_predictions is a list of multi-step forecasts
    """
    # Load pretrained small model
    model = AuroraSmall()
    model.load_checkpoint("microsoft/aurora", "aurora-0.25-small-pretrained.ckpt")
    print("Model loaded successfully!")

    print("Batch shapes:")
    print(f"- Surface vars: {batch.surf_vars['2t'].shape}")
    print(f"- Static vars: {batch.static_vars['z'].shape}")
    print(f"- Atmospheric vars: {batch.atmos_vars['t'].shape}")

    # Run model on CPU or GPU based on availability
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = model.to(device)
    batch = batch.to(device)

    # Get prediction
    with torch.inference_mode():
        prediction = model.forward(batch)
        rollout_preds = [pred.to("cpu") for pred in rollout(model, batch, steps=steps)]

    # Move results back to CPU
    prediction = prediction.to("cpu")

    print(f"Prediction completed with shape: {prediction.surf_vars['2t'].shape}")
    print(f"Generated {len(rollout_preds)} rollout predictions")

    return prediction, rollout_preds

In [23]:
def visualize_aurora_predictions(batch, prediction, rollout_preds=None):
    """
    Visualize Aurora model input and predictions with clearer temporal evolution.

    Args:
        batch (Batch): The input batch for Aurora model
        prediction (Batch): The model's prediction
        rollout_preds (list, optional): List of rollout prediction batches
    """
    # Get the number of rollout steps
    num_steps = 1 + (len(rollout_preds) if rollout_preds else 0)

    # Create a figure for each variable type to show evolution
    variables_to_plot = {
        "Surface Temperature": {"var_dict": "surf_vars", "var_name": "2t", "level_idx": None},
        "Surface Wind U": {"var_dict": "surf_vars", "var_name": "10u", "level_idx": None},
        "Surface Wind V": {"var_dict": "surf_vars", "var_name": "10v", "level_idx": None},
        "Mean Sea Level Pressure": {"var_dict": "surf_vars", "var_name": "msl", "level_idx": None},
    }

    # Add atmospheric variables at different levels
    if batch.atmos_vars and len(batch.metadata.atmos_levels) > 0:
        # Add upper and lower atmosphere temperature
        if len(batch.metadata.atmos_levels) >= 2:
            variables_to_plot[f"Temp at {batch.metadata.atmos_levels[0]} hPa"] = {
                "var_dict": "atmos_vars",
                "var_name": "t",
                "level_idx": 0,
            }
            variables_to_plot[f"Temp at {batch.metadata.atmos_levels[-1]} hPa"] = {
                "var_dict": "atmos_vars",
                "var_name": "t",
                "level_idx": -1,
            }

    # Plot each variable's evolution
    for var_title, var_info in variables_to_plot.items():
        var_dict = var_info["var_dict"]
        var_name = var_info["var_name"]
        level_idx = var_info["level_idx"]

        # Skip if variable doesn't exist
        if not hasattr(batch, var_dict) or var_name not in getattr(batch, var_dict):
            continue

        plt.figure(figsize=(15, 4))
        plt.suptitle(f"Evolution of {var_title}")

        # Plot input timesteps (t-1, t)
        for i in range(2):
            plt.subplot(1, 2 + num_steps, i + 1)
            if level_idx is None:
                # Surface variable
                data = getattr(batch, var_dict)[var_name][0, i].numpy()
                plt.title(f"Input t-{1-i}")
            else:
                # Atmospheric variable with vertical level
                data = getattr(batch, var_dict)[var_name][0, i, level_idx].numpy()
                plt.title(f"Input t-{1-i}")

            im = plt.imshow(data)
            plt.colorbar(im)

        # Plot immediate prediction (t+1)
        plt.subplot(1, 2 + num_steps, 3)
        if level_idx is None:
            # Surface variable
            data = getattr(prediction, var_dict)[var_name][0, 0].numpy()
            plt.title("Forecast t+1")
        else:
            # Atmospheric variable with vertical level
            data = getattr(prediction, var_dict)[var_name][0, 0, level_idx].numpy()
            plt.title("Forecast t+1")

        im = plt.imshow(data)
        plt.colorbar(im)

        # Plot rollout predictions if available (t+2, t+3, ...)
        if rollout_preds:
            for i, pred in enumerate(rollout_preds):
                plt.subplot(1, 2 + num_steps, 4 + i)
                if level_idx is None:
                    # Surface variable
                    data = getattr(pred, var_dict)[var_name][0, 0].numpy()
                    plt.title(f"Forecast t+{i+2}")
                else:
                    # Atmospheric variable with vertical level
                    data = getattr(pred, var_dict)[var_name][0, 0, level_idx].numpy()
                    plt.title(f"Forecast t+{i+2}")

                im = plt.imshow(data)
                plt.colorbar(im)

        plt.tight_layout()
        plt.show()

    # Add a special plot for vertical cross-section of temperature if we have atmos variables
    if batch.atmos_vars and "t" in batch.atmos_vars and len(batch.metadata.atmos_levels) > 1:
        plt.figure(figsize=(15, 8))
        plt.suptitle("Vertical Cross-section of Temperature (middle latitude)")

        # Get middle latitude index
        mid_lat = batch.atmos_vars["t"].shape[3] // 2

        # Plot input timesteps
        for i in range(2):
            plt.subplot(1, 2 + num_steps, i + 1)
            data = batch.atmos_vars["t"][0, i, :, mid_lat, :].numpy()
            plt.title(f"Input t-{1-i}")
            im = plt.imshow(data, aspect="auto", origin="lower")
            plt.ylabel("Pressure Level")
            plt.xlabel("Longitude")
            plt.colorbar(im)

        # Plot immediate prediction
        plt.subplot(1, 2 + num_steps, 3)
        data = prediction.atmos_vars["t"][0, 0, :, mid_lat, :].numpy()
        plt.title("Forecast t+1")
        im = plt.imshow(data, aspect="auto", origin="lower")
        plt.ylabel("Pressure Level")
        plt.xlabel("Longitude")
        plt.colorbar(im)

        # Plot rollout predictions if available
        if rollout_preds:
            for i, pred in enumerate(rollout_preds):
                plt.subplot(1, 2 + num_steps, 4 + i)
                data = pred.atmos_vars["t"][0, 0, :, mid_lat, :].numpy()
                plt.title(f"Forecast t+{i+2}")
                im = plt.imshow(data, aspect="auto", origin="lower")
                plt.ylabel("Pressure Level")
                plt.xlabel("Longitude")
                plt.colorbar(im)

        plt.tight_layout()
        plt.show()

In [ ]:
if __name__ == "__main__":
    # Set random seed for reproducibility
    torch.manual_seed(42)

    # Step 1: Create a batch with custom dimensions
    print("Step 1: Creating Aurora batch...")
    grid_height = 32
    grid_width = 32
    num_levels = 20
    batch = create_aurora_batch(grid_height, grid_width, num_levels)

    # Step 2: Run the Aurora model
    print("\nStep 2: Running Aurora model...")
    rollout_steps = 3
    prediction, rollout_preds = run_aurora_model(batch, steps=rollout_steps)
    print(f"Model execution completed, generated {len(rollout_preds)} rollout steps")

    # Step 3: Visualize the results
    print("\nStep 3: Visualizing predictions...")
    visualize_aurora_predictions(batch, prediction, rollout_preds)
    print("Visualization complete")

    print("\nAll steps completed successfully!")

Step 1: Creating Aurora batch...
Created batch with grid size 32x32 and 20 pressure levels

Step 2: Running Aurora model...
Model loaded successfully!
Batch shapes:
- Surface vars: torch.Size([1, 2, 32, 32])
- Static vars: torch.Size([32, 32])
- Atmospheric vars: torch.Size([1, 2, 20, 32, 32])
Using device: cuda


KeyError: 'z_350'

# test with our sample data